Multi-Agent Role Interaction (Reviewer ➝ Editor ➝ Finalizer)

## Question 1: Multi-Agent Role Interaction (Reviewer ➝ Editor ➝ Finalizer)

Task: Implement a three-agent pipeline where each agent performs a unique role. Pass the message correctly to avoid format errors.


A three-agent pipeline consists of three specialized agents:

Reviewer Agent – Reviews the content and identifies mistakes.

Editor Agent – Corrects grammar, spelling, and improves readability.

Finalizer Agent – Formats and prepares the final polished output.

    User
      │
      ▼
    Reviewer
      │
      ▼
    Editor
      │
      ▼
    Finalizer
      │
      ▼
    Final Output

without LLM

In [13]:
# ============================================================
# Multi-Agent Role Interaction (Without LLM)
# Reviewer ➝ Editor ➝ Finalizer
# Google Colab Version
# ============================================================

class ReviewerAgent:
    def review(self, message):
        print("Reviewer Agent Received:")
        print(message)

        # Review comments
        reviewed_message = {
            "content": message,
            "review_comment": "Grammar and clarity need improvement."
        }

        print("\nReviewer Output:")
        print(reviewed_message)

        return reviewed_message


class EditorAgent:
    def edit(self, reviewed_data):
        print("\nEditor Agent Received:")
        print(reviewed_data)

        original = reviewed_data["content"]

        # Simple editing
        edited = original.replace("AI", "Artificial Intelligence")
        edited = edited.strip().capitalize()

        editor_output = {
            "edited_content": edited,
            "review_comment": reviewed_data["review_comment"]
        }

        print("\nEditor Output:")
        print(editor_output)

        return editor_output


class FinalizerAgent:
    def finalize(self, editor_data):
        print("\nFinalizer Agent Received:")
        print(editor_data)

        final_output = f"""
==============================
FINAL DOCUMENT
==============================
{editor_data['edited_content']}

Review Note:
{editor_data['review_comment']}
==============================
"""

        return final_output


# ============================================================
# Pipeline
# ============================================================

reviewer = ReviewerAgent()
editor = EditorAgent()
finalizer = FinalizerAgent()

input_message = "ai is changing the world."

reviewed = reviewer.review(input_message)
edited = editor.edit(reviewed)
final_document = finalizer.finalize(edited)

print(final_document)

Reviewer Agent Received:
ai is changing the world.

Reviewer Output:
{'content': 'ai is changing the world.', 'review_comment': 'Grammar and clarity need improvement.'}

Editor Agent Received:
{'content': 'ai is changing the world.', 'review_comment': 'Grammar and clarity need improvement.'}

Editor Output:
{'edited_content': 'Ai is changing the world.', 'review_comment': 'Grammar and clarity need improvement.'}

Finalizer Agent Received:
{'edited_content': 'Ai is changing the world.', 'review_comment': 'Grammar and clarity need improvement.'}

FINAL DOCUMENT
Ai is changing the world.

Review Note:
Grammar and clarity need improvement.



with GROQ LLM

In [14]:
!pip -q install groq

In [15]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

print("API Loaded Successfully")

API Loaded Successfully


In [16]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

MODEL = "llama-3.3-70b-versatile"

Create Agent Function

In [17]:
def run_agent(system_prompt, user_message):

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_message
            }
        ]
    )

    return response.choices[0].message.content

Reviewer Agent

In [18]:
reviewer_prompt = """
You are a Reviewer.

Your job:
- Review the text.
- Find grammar mistakes.
- Find clarity issues.
- Give suggestions.

Return only the reviewed text and suggestions.
"""

input_message = """
AI change world fast and many peoples use it daily.
"""

review_output = run_agent(reviewer_prompt, input_message)

print("===== REVIEWER OUTPUT =====\n")
print(review_output)

===== REVIEWER OUTPUT =====

AI is changing the world fast and many people use it daily.

Suggestions:
- "AI change" should be "AI is changing" to maintain subject-verb agreement and to use the correct verb tense.
- "peoples" should be "people" as "people" is the correct plural form of the noun.


Editor Agent

In [19]:
editor_prompt = """
You are an Editor.

Your job:
- Read the reviewer comments.
- Improve grammar.
- Improve clarity.
- Rewrite professionally.

Return only the edited paragraph.
"""

editor_input = f"""
Original Text:

{input_message}

Reviewer Notes:

{review_output}
"""

edited_output = run_agent(editor_prompt, editor_input)

print("===== EDITOR OUTPUT =====\n")
print(edited_output)

===== EDITOR OUTPUT =====

AI is changing the world rapidly, and numerous individuals utilize it on a daily basis.


Finalizer Agent

In [20]:
finalizer_prompt = """
You are a Finalizer.

Your task:
- Produce the final polished document.
- Remove unnecessary comments.
- Output only the final version.
"""

final_output = run_agent(finalizer_prompt, edited_output)

print("===== FINAL OUTPUT =====\n")
print(final_output)

===== FINAL OUTPUT =====

Artificial intelligence is revolutionizing the world at an unprecedented pace, and its impact is being felt by countless individuals who incorporate it into their daily lives. From virtual assistants and smart home devices to personalized product recommendations and autonomous vehicles, AI is transforming the way people live, work, and interact with one another. As AI technology continues to advance and improve, it is likely to have an even more profound effect on society, driving innovation, increasing efficiency, and redefining the boundaries of what is possible.


                User Input
                     │
                     ▼
             Reviewer Agent
                     │
          Review + Suggestions
                     │
                     ▼
              Editor Agent
                     │
             Edited Version
                     │
                     ▼
            Finalizer Agent
                     │
                     ▼
          Final Polished Output